In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the webpage
url = 'https://www.pff.com/news/nfl-roster-rankings-all-32-teams-2025-strengths-weaknesses-x-factors'

# Fetch the webpage content
response = requests.get(url)
if response.status_code == 200:
    page_content = response.content
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")

# Parse the webpage content
soup = BeautifulSoup(page_content, 'html.parser')

# Initialize an empty list to store the data
data = []

# Find all team sections (h3 headers) and their corresponding tables
team_headers = soup.find_all('h3')
tables = soup.find_all('table')

# Each team has one table, match them by order
for header, player_table in zip(team_headers, tables):
    team_name = header.text.strip()
    rows = player_table.find_all('tr')
    for row in rows[1:]:  # Skip the header row (OFFENSE/DEFENSE)
        cells = row.find_all('td')
        if len(cells) == 2:
            for cell in cells:
                cell_text = cell.text.strip()
                if cell_text:
                    parts = cell_text.split(' ')
                    position = parts[0]
                    player_name = ' '.join(parts[1:-1])
                    grade = parts[-1].strip('()')
                    data.append({
                        'Team': team_name,
                        'Position': position,
                        'Player': player_name,
                        'Grade': grade
                    })

# Create a DataFrame from the extracted data
currAVs = pd.DataFrame(data)

currAVs['Grade'] = currAVs['Grade'].str.replace('*', '')
currAVs['Grade'] = currAVs['Grade'].astype(float)

# Display the DataFrame
currAVs

,Team,Position,Player,Grade
0,1. SAN FRANCISCO 49ERS,QB,Brock Purdy,87.4
1,1. SAN FRANCISCO 49ERS,DI,Javon Hargrave,74.9
2,1. SAN FRANCISCO 49ERS,RB,Christian McCaffrey,90.3
3,1. SAN FRANCISCO 49ERS,DI,Maliek Collins,60.9
4,1. SAN FRANCISCO 49ERS,RB,Jordan Mason,84.7
...,...,...,...,...
763,28. ARIZONA CARDINALS,CB,Andru Phillips,71.7
764,28. ARIZONA CARDINALS,RG,Aaron Stinnie,56.5
765,28. ARIZONA CARDINALS,S,Dane Belton,51.7
766,28. ARIZONA CARDINALS,RT,Jermaine Eluemunor,68.7


In [5]:
import pandas as pd

currAVs['Grade'] = pd.to_numeric(currAVs['Grade'], errors='coerce')

# Define position groups
position_groups = {
    'oline': ['LT', 'LG', 'C', 'RG', 'RT'],
    'qb': ['QB'],
    'rb': ['RB'],
    'wrte': ['WR', 'TE'],
    'dst': ['Edge', 'LB', 'Dl', 'CB', 'S']
}

# Initialize a list to store the data for the new DataFrame
new_data = []

# Get unique teams
teams = currAVs['Team'].unique()

# Calculate the averages for each team
for team in teams:
    team_data = {'Team': team}
    team_df = currAVs[currAVs['Team'] == team]
    for group, positions in position_groups.items():
        group_grades = team_df[team_df['Position'].isin(positions)]['Grade']
        if not group_grades.empty:
            team_data[group] = group_grades.mean()
        else:
            team_data[group] = None
    new_data.append(team_data)

# Create the new DataFrame
currAVs = pd.DataFrame(new_data)

# Normalize team names to title case (PFF page may return uppercase)
currAVs['Team'] = currAVs['Team'].str.title()

corrections = {
    '1. San Francisco 49Ers': 'SF',
    '1. San Francisco 49ers': 'SF',
    '2. Kansas City Chiefs': 'KC',
    '3. Philadelphia Eagles': 'PHI',
    '4. New York Jets': 'NYJ',
    '5. Baltimore Ravens': 'BAL',
    '6. Detroit Lions': 'DET',
    '7. Houston Texans': 'HOU',
    '8. Cincinnati Bengals': 'CIN',
    '9. Dallas Cowboys': 'DAL',
    '10. Buffalo Bills': 'BUF',
    '11. Miami Dolphins': 'MIA',
    '12. Cleveland Browns': 'CLE',
    '13. Green Bay Packers': 'GB',
    '14. Los Angeles Rams': 'LA',
    '15. Atlanta Falcons': 'ATL',
    '16. Pittsburgh Steelers': 'PIT',
    '17. Seattle Seahawks': 'SEA',
    '18. Tampa Bay Buccaneers': 'TB',
    '19. Jacksonville Jaguars': 'JAX',
    '20. Chicago Bears': 'CHI',
    '21. Minnesota Vikings': 'MIN',
    '22. Indianapolis Colts': 'IND',
    '23. Las Vegas Raiders': 'LV',
    '24. New Orleans Saints': 'NO',
    '25. Tennessee Titans': 'TEN',
    '26. Los Angeles Chargers': 'LAC',
    '27. Washington Commanders': 'WAS',
    '28. Arizona Cardinals': 'ARI',
    '29. New England Patriots': 'NE',
    '30. Carolina Panthers': 'CAR',
    '31. New York Giants': 'NYG',
    '32. Denver Broncos': 'DEN'
}

currAVs['Team'] = currAVs['Team'].replace(corrections)
currAVs

,Team,oline,qb,rb,wrte,dst
0,SF,71.480,87.400,87.500,84.67500,75.690000
1,KC,69.640,90.500,72.700,75.87500,67.510000
2,PHI,69.780,86.700,62.750,73.52500,72.410000
3,NYJ,70.600,39.400,78.650,67.32500,76.550000
4,,70.715,84.375,75.325,70.69375,68.900000
5,BAL,81.840,85.700,77.950,76.62500,67.770000
6,DET,57.200,83.100,68.500,81.35000,72.500000
7,HOU,67.380,77.900,65.400,73.20000,67.440000
8,CIN,62.420,90.000,67.100,72.67500,75.600000
9,DAL,64.120,86.900,88.400,78.20000,71.700000


In [6]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the second pickle file
file_path = '../PickleFiles/AVbyPositionGroup.pkl'
av_by_position_group = pd.read_pickle(file_path)

# Display the second DataFrame to understand its structure
print(av_by_position_group.head())

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Define columns to be scaled
columns_to_scale = ['oline', 'qb', 'rb', 'wrte', 'dst']

# Fit and transform the currAVs DataFrame
currAVs_scaled = currAVs.copy()
currAVs_scaled[columns_to_scale] = scaler.fit_transform(currAVs[columns_to_scale])

# Fit and transform the av_by_position_group DataFrame
av_by_position_group_scaled = av_by_position_group.copy()
av_by_position_group_scaled[columns_to_scale] = scaler.fit_transform(av_by_position_group[columns_to_scale])

# Display the scaled DataFrames to verify
print(currAVs_scaled.head())
print(av_by_position_group_scaled.head())
currAVs_scaled = currAVs_scaled.rename(columns={'Team': 'team'})

# Save the scaled DataFrames to pickle files
currAVs_scaled.to_pickle('../PickleFiles/currAVs.pkl')
av_by_position_group_scaled.to_pickle('../PickleFiles/AVbyPositionGroup.pkl')

  team      oline         qb         rb       wrte         dst  season
0  ARI  45.754273  12.033595   8.622070  34.249463  114.158099    2013
1  ATL  42.615458  11.595115   6.541953  33.001481   82.744086    2013
2  BAL  38.631576  10.237457   6.983059  29.137376  105.825745    2013
3  BUF  40.925326   9.685038  11.860243  27.565110   96.084038    2013
4  CAR  44.184865  10.772806  11.587971  30.661062  145.110682    2013
  Team     oline        qb        rb      wrte       dst
0   SF  0.579545  0.867993  0.964912  0.996143  0.941056
1   KC  0.504870  0.924051  0.387914  0.656702  0.380398
2  PHI  0.510552  0.855335  0.000000  0.566056  0.716244
3  NYJ  0.543831  0.000000  0.619883  0.326905  1.000000
4       0.548498  0.813291  0.490253  0.456847  0.475668
  team     oline        qb        rb      wrte       dst  season
0  ARI  0.403897  0.409928  0.188717  0.409928  0.576022    2013
1  ATL  0.335620  0.375073  0.068753  0.375073  0.202355    2013
2  BAL  0.248962  0.267154  0.094193 